In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns 
import matplotlib.pyplot as plt
import matplotlib
import pickle
import os
import glob

import pingouin as pg

from sklearn.metrics.pairwise import euclidean_distances
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'


In [ ]:
# set directories and load data
basedir = '/path/to/pain-induced-manifold-changes'
datdir = os.path.join(basedir, 'data')


file = os.path.join(datdir, 'pain-induced-manifold-changes_10categ_data_prep_rsa.pickle')
with open(file, 'rb') as f:
    dat_all = pickle.load(f)
    
sub_ids = list(dat_all.keys())

categ_fname = os.path.join(datdir, 'category_names.pickle')
with open(categ_fname, 'rb') as f:
    categ_names = pickle.load(f)



In [ ]:
# Perform RSA - based on image trials

rois = ['V1', 'V2', 'hV4', 'LO']
levels = [0, 1, 2]
cond_names = ['Warmth', 'LowHeat', 'HighHeat']
categ_order = ['bird', 'insect', 'reptile', 'seacreature', 'wildcat', 'clothes', 'furniture', 'instrument', 'meal', 'transport']


within_corr_df = pd.DataFrame({'Subject': [],
                               'ROI': [],
                               'Condition': [],
                               'Category': [],
                               'corr_within': []})
btw_corr_df = pd.DataFrame({'Subject': [],
                            'ROI': [],
                            'Condition': [],
                            'Category1': [],
                            'Category2': [],
                            'corr_btw': []})
                            
for sub in sub_ids: # loop through subjects
 
    sub_dat = dat_all[sub]
    
    for reg in rois: # loop through ROIs
     
        dat_temp = sub_dat[reg]
    
      
        for cond in levels: # loop through heat levels

            print(f'Working on {sub} | {reg} | {cond_names[cond]}')
            dat = dat_temp[cond]
            
            mean_repre = {}
            for cat_n, cat in enumerate(categ_names): # find mean representation of each category

                mean_repre_temp = np.mean(dat[cat_n,:,:],axis=1)
                mean_repre[cat] = mean_repre_temp
            
            # RSA between categories - using categ order for easier visualisation of animate vs. inanimate
            for cat1_n, cat1 in enumerate(categ_order):
                for cat2_n, cat2 in enumerate(categ_order):
                    if cat1_n<cat2_n:
                        btw_corr_temp = np.corrcoef(mean_repre[cat1], mean_repre[cat2])[0][1]
                        
                        btw_corr_df_temp = pd.DataFrame({'Subject': [sub],
                                                         'ROI': [reg],
                                                         'Condition': [cond_names[cond]],
                                                         'Category1': [cat1],
                                                         'Category2': [cat2],
                                                         'corr_btw': [btw_corr_temp]})
                        btw_corr_df = pd.concat([btw_corr_df, btw_corr_df_temp],ignore_index=True)
                

In [ ]:
# Get mean values for boxplots

btw_corr_df_mean = btw_corr_df.groupby(['Subject','ROI','Condition'],sort=False).mean(numeric_only=True).reset_index()


In [ ]:
# get mean for heatmap
df_heat = btw_corr_df.copy()
categ_order1 = btw_corr_df['Category1'].unique()
categ_order2 = btw_corr_df['Category2'].unique()

df_heat.Category1 = pd.Categorical(df_heat.Category1,
                                  categories = categ_order1,
                                  ordered = True)
df_heat.Category2 = pd.Categorical(df_heat.Category2,
                                  categories = categ_order2,
                                  ordered = True)

df_heat = (
    df_heat.groupby(['ROI', 'Condition', 'Category1', 'Category2'], sort=False)
    .mean(numeric_only=True)
    .reset_index()
)


In [ ]:
# Plotting
save_figure = 0
fig_type = 'heatmap' #'heatmap' # 'main_btw', 


if fig_type == 'main_btw':
    dat_to_plot = btw_corr_df_mean
    cols = ["#fee0d2", "#fc9272", "#de2d26"]
    title = "Between-category similarity"
    ax = sns.boxplot(data=dat_to_plot, x="ROI", y="corr_btw",
                    hue="Condition", showcaps=False,
                    flierprops={"marker":""},
                    palette= cols, order=rois, hue_order=cond_names)  
    sns.stripplot(data=dat_to_plot, x="ROI", y="corr_btw",
                  hue="Condition", dodge=True,palette=cols,linewidth=0.5,edgecolor='gray', order=rois,hue_order=cond_names) 
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[0:3], labels[0:3], bbox_to_anchor=(1.05,1), loc=2, borderaxespad=0.)
    ax.set_title(title)
    ax.set_ylabel("Similarity (Pearson's r)")
    fig = ax.get_figure()
    sns.despine()

    if save_figure:
        fig.savefig(os.path.join(figdir, f'pacman_240306_mni_rsa_{roi_type}_{fig_type}.pdf'), bbox_inches='tight')     
        
elif fig_type == 'heatmap':
    dat_to_plot = df_heat
    def draw_heatmap(*args,**kwargs):
        data = kwargs.pop('data')
        d = data.pivot_table(index=args[1], columns=args[0],values=args[2], sort=False)
        sns.heatmap(d, **kwargs)

    g = sns.FacetGrid(dat_to_plot, col='ROI', col_order=rois, row = 'Condition', row_order = cond_names) #, row = 'Condition', row_order = condition

    cbar_ax = g.fig.add_axes([1, .2, .03, .7])
    g = g.map_dataframe(draw_heatmap, 'Category1', 'Category2', 'corr_btw',annot=False, vmin=0.2, vmax=0.8,cbar_ax=cbar_ax, square=True)

    g.set_axis_labels('','')

    if save_figure:
        plt.savefig(os.path.join(figdir, f'pacman_240306_mni_rsa_{roi_type}_{fig_type}.pdf'), bbox_inches='tight')

In [ ]:
# ANOVA and post hoc
# ---------- shared formatting helpers ----------

def to_superscript(n):
    """Convert an integer to unicode superscript (e.g., -26 -> ⁻²⁶)."""
    sup = str.maketrans('0123456789-', '⁰¹²³⁴⁵⁶⁷⁸⁹⁻')
    return str(n).translate(sup)


def fmt_p_num(x):
    """p-value to three decimals; scientific notation (n.nnn × 10⁻ⁿ)
    when p < .001. Blank for NaN."""
    if pd.isna(x):
        return ''
    if x >= .001:
        return f'{x:.3f}'.lstrip('0')
    mantissa, exp = f'{x:.3e}'.split('e')
    return f'{mantissa} × 10{to_superscript(int(exp))}'



# ---------- ANOVA table ----------

def rm_anova_paper_table_combined(data, dv='mean_acc', subject='sub',
                                  within=('cond', 'roi'), labels=None):
    """pingouin two-way rm ANOVA -> publication-ready table.
    df/SS/MS are uncorrected (preserves additivity); ε and GG-corrected
    p-values are reported as additional columns."""
    aov = pg.rm_anova(data=data, dv=dv, within=list(within),
                      subject=subject, detailed=True)
    aov.columns = [c.replace('-', '_') for c in aov.columns]  # unify p-unc / p_unc

    if labels is None:
        labels = {'cond': 'Condition', 'roi': 'ROI',
                  'cond * roi': 'Condition × ROI'}

    # Subject and Total terms, computed directly from the raw data
    grand = data[dv].mean()
    ss_total = ((data[dv] - grand) ** 2).sum()
    df_total = len(data) - 1
    sub_means = data.groupby(subject)[dv].mean()
    n_cells = data.groupby(subject).size().iloc[0]  # cells per subject (12 here)
    ss_sub = (n_cells * (sub_means - grand) ** 2).sum()
    df_sub = sub_means.size - 1

    blank = dict(F=np.nan, p_unc=np.nan, eps=np.nan,
                 p_GG=np.nan, eta2G=np.nan)
    rows = [dict(Source='Subject', SS=ss_sub, df=df_sub,
                 MS=ss_sub / df_sub, **blank)]

    for _, r in aov.iterrows():
        lab = labels.get(r['Source'], r['Source'])
        ms_err = r['MS'] / r['F']      # recover error term from F = MS_effect / MS_error
        ss_err = ms_err * r['ddof2']
        rows.append(dict(Source=lab, SS=r['SS'], df=int(r['ddof1']),
                         MS=r['MS'], F=r['F'], p_unc=r['p_unc'],
                         eps=r['eps'], p_GG=r['p_GG_corr'],
                         eta2G=r['ng2']))
        rows.append(dict(Source=f'Error ({lab})', SS=ss_err,
                         df=int(r['ddof2']), MS=ms_err, **blank))

    rows.append(dict(Source='Total', SS=ss_total, df=df_total,
                     MS=np.nan, **blank))
    table = pd.DataFrame(rows)

    # Sanity check: both SS and df should sum to the Total row
    body = table[table['Source'] != 'Total']
    if not np.isclose(body['SS'].sum(), ss_total, rtol=1e-3):
        print(f"Warning: sum of SS ({body['SS'].sum():.4f}) "
              f"!= SS_total ({ss_total:.4f})")
    if body['df'].sum() != df_total:
        print(f"Warning: sum of df ({body['df'].sum():.0f}) "
              f"!= df_total ({df_total})")

    # Formatting for publication
    fmt = table.copy()
    fmt['SS'] = fmt['SS'].map(lambda x: f'{x:.3f}')
    fmt['df'] = fmt['df'].map(lambda x: f'{x:.0f}')
    fmt['MS'] = fmt['MS'].map(lambda x: '' if pd.isna(x) else f'{x:.4f}')
    fmt['F'] = fmt['F'].map(lambda x: '' if pd.isna(x) else f'{x:.2f}')
    fmt['p_unc'] = fmt['p_unc'].map(fmt_p_num)
    fmt['eps'] = fmt['eps'].map(lambda x: '' if pd.isna(x)
                                else f'{x:.3f}'.lstrip('0'))
    fmt['p_GG'] = fmt['p_GG'].map(fmt_p_num)
    fmt['eta2G'] = fmt['eta2G'].map(lambda x: '' if pd.isna(x)
                                    else f'{x:.3f}'.lstrip('0'))
    fmt = fmt.rename(columns={'p_unc': 'p', 'sig_unc': '',
                              'eps': 'ε', 'p_GG': 'p (GG)',
                              'sig_GG': ' ', 'eta2G': 'η²G'})
    # column order: p right before its flag column
    fmt = fmt[['Source', 'SS', 'df', 'MS', 'F', 'p',
               'ε', 'p (GG)', 'η²G']]
    return table, fmt


# ---------- Post hoc table ----------

def posthoc_paper_table(data, dv='mean_acc', subject='sub',
                        factor='cond', padjust='fdr_bh', labels=None):
    """Pairwise post hoc comparisons for one within-subject factor
    (paired t-tests with multiple-comparison correction),
    formatted as a publication-ready table."""
    # Collapse across the other factor: one value per subject x level
    agg = data.groupby([subject, factor])[dv].mean().reset_index()
    wide = agg.pivot(index=subject, columns=factor, values=dv)

    ph = pg.pairwise_tests(data=agg, dv=dv, within=factor,
                           subject=subject, padjust=padjust,
                           effsize='cohen')
    ph.columns = [c.replace('-', '_') for c in ph.columns] 
    # Mean difference (A - B) and paired 95% CI, per comparison
    stats_rows = []
    for _, r in ph.iterrows():
        d = wide[r['A']] - wide[r['B']]
        se = d.std(ddof=1) / np.sqrt(len(d))
        tcrit = stats.t.ppf(0.975, len(d) - 1)
        stats_rows.append((d.mean(), d.mean() - tcrit * se,
                           d.mean() + tcrit * se))
    ph[['mean_diff', 'ci_low', 'ci_high']] = pd.DataFrame(
        stats_rows, index=ph.index)

    # Optional relabeling (e.g., 0/1/2 -> condition names)
    if labels:
        ph['A'] = ph['A'].map(labels).fillna(ph['A'])
        ph['B'] = ph['B'].map(labels).fillna(ph['B'])

    fmt = pd.DataFrame({
        'Comparison': ph['A'].astype(str) + ' vs. ' + ph['B'].astype(str),
        'Mean diff.': ph['mean_diff'].map(lambda x: f'{x:.4f}'),
        '95% CI': ph.apply(lambda r: f"[{r['ci_low']:.4f}, {r['ci_high']:.4f}]",
                           axis=1),
        't': ph['T'].map(lambda x: f'{x:.2f}'),
        'df': ph['dof'].map(lambda x: f'{x:.0f}'),
        'p': ph['p_unc'].map(fmt_p_num),
        f'p ({padjust})': ph['p_corr'].map(fmt_p_num),
        "Cohen's d": ph['cohen'].map(lambda x: f'{x:.2f}'),
    })
    return ph, fmt

cond_labels = {'Warmth': 'Warmth', 'LowHeat': 'Low heat', 'HighHeat': 'High heat'}
raw_aov, aov_table = rm_anova_paper_table_combined(btw_corr_df_mean, dv='corr_btw', subject='Subject',
                                                   within=('Condition', 'ROI'),
                                                   labels = {'Condition': 'Heat level', 'ROI': 'ROI',
                                                             'Condition * ROI': 'Heat level × ROI'})
raw_cond, cond_table = posthoc_paper_table(btw_corr_df_mean, dv='corr_btw', subject='Subject',
                                           factor='Condition', padjust='fdr_bh', 
                                           labels=cond_labels)
raw_roi, roi_table = posthoc_paper_table(btw_corr_df_mean, dv='corr_btw', subject='Subject',
                                         factor='ROI', padjust='fdr_bh',
                                         labels=cond_labels)

In [ ]:
# test simple effects of heat level within each ROI
all_results = []
for roi in btw_corr_df_mean['ROI'].unique():
    print(f"\n--- Simple effect of Heat level within ROI = {roi} ---")
    subset = btw_corr_df_mean[btw_corr_df_mean['ROI'] == roi]
    raw, tbl = posthoc_paper_table(subset, dv='corr_btw', subject='Subject',
                                    factor='Condition', padjust='fdr_bh', labels=cond_labels)
    tbl.style.hide(axis='index')
    raw['ROI'] = roi  # tag which ROI this comparison came from
    all_results.append(raw)


# Combine all 12 comparisons into one dataframe
combined = pd.concat(all_results, ignore_index=True)

# Apply FDR correction across all 12 uncorrected p-values
combined['p_corr_global'] = fdrcorrection(combined['p_unc'], method='indep')[1]

print(combined[['ROI', 'A', 'B', 'T', 'dof', 'p_unc', 'p_corr_global']].to_string(index=False))